# KALIA bench v0.2.0 — the registered evaluation suite

Runs the same five zero-shot tasks as v0.1.2 (ARC-Easy, PIQA, HellaSwag, LAMBADA,
WinoGrande; 0-shot, 500 samples) against the **v0.2.0** checkpoint, so the two
versions are finally comparable on the same suite.

CPU kernel: no GPU quota. Sourced from the `kalia-train-v020` output; the code
dataset must be the latest version, since it supplies `eval_bench.py` and the fix
that makes `find_ckpt` prefer the attached training kernel.

In [ ]:
# lm-eval 0.4.9 imports transformers.AutoModelForVision2Seq, which was REMOVED in
# transformers v5 -- and Kaggle's image now ships v5. Pinning transformers<5 as well
# gives a pair that matches. kalia_lm.py does not import transformers at all (it uses
# tiktoken + our own model.py), so the downgrade cannot affect our code.
!pip install -q "lm-eval==0.4.9" "transformers<5" tiktoken pyyaml datasets
import glob, importlib.metadata as md, shutil, sys
print(sys.version)
import importlib.metadata as md
for pkg in ('lm_eval', 'transformers', 'torch', 'datasets'):
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except Exception as exc:
        print(f"{pkg:14s} MISSING ({exc})")
import transformers
print("AutoModelForVision2Seq present:", hasattr(transformers, "AutoModelForVision2Seq"))
print(''.join(sorted(glob.glob('/kaggle/input/notebooks/subhajitlucky/kalia-train-v020/*'))))

In [ ]:
code = sorted(glob.glob("/kaggle/input/datasets/subhajitlucky/kalia-code-dev"))
assert code, "attach the kalia-code-dev dataset"
root = code[0]
for f in ["eval_bench.py", "kalia_lm.py", "model.py", "data.py", "eval_reversibility.py"]:
    src = f"{root}/{f}"
    shutil.copy(src, "/kaggle/working/")
    print("copied", f)
ckpts = sorted(glob.glob("/kaggle/input/notebooks/subhajitlucky/kalia-train-v020/**/ckpt.pt", recursive=True))
print("checkpoints found:", ckpts)

In [ ]:
import subprocess
subprocess.run(
    [sys.executable, "eval_bench.py",
     "--ckpt", ckpts[0],
     "--out", "/kaggle/working/bench_v020.json",
     "--limit", "500"],
    check=True,
)

In [ ]:
!cat /kaggle/working/bench_v020.json

print("\n=== comparison target: v0.1.2 (published, same suite) ===")
print("PIQA 61.4 | ARC-Easy 45.8 | HellaSwag 36.8 | WinoGrande 50.2 | LAMBADA 23.0")

Results are reported as measured, including any regression. Per the registered
evaluation, the yardstick is the v2b val set and the comparator is v0.1.2 at 3.0533
on that set; the pre-registered secondary metric S-A allows no task to regress by
more than 1.0 pp.